# 🤔 I THINK ULTIMATELY MAKE THIS ONE NOTEBOOK? 

# EIA Thermoelectric Data Ingestion (Phase 1)

**Author:** Amy Zhang  
**AI Collaborators:** ChatGPT, Claude Code, Perplexity AI  
**Date:** January 2026

## Table of Contents
  1. Import Libraries
  2. Integrate EIA Thermoelectric Cooling Dataset
  3. Preliminary Dataset Polish for SQL Import
  4. Persist SQL-Ready 'Ground Truth' Dataset

# 1. Import Libraries 
# EDIT -- you don't need all these libraries for THIS notebook!

In [1]:
# 1. Auto-reload external modules (during development)
%load_ext autoreload
%autoreload 2

# 2. Standard library
import os
import re

# 3. Third party  
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.colors import ListedColormap


# 2. Integrate  EIA Thermoelectric Cooling Dataset

In [2]:
from scripts.dataset_pipeline import datasets_overview, merge_folder_files

In [3]:
# Test fallback
datasets_overview()


🔍 PHASE 1: DATASET OVERVIEW
   Target folder: /Users/amyzhang/Documents/Projects_2026/eia_2.0/datasets_thermoelectric
   This step inspects schemas and estimates memory usage only.
   No files are merged in Phase 1.

   • Cooling_Boiler_Generator_Data_Detail_2022.xlsx | columns: 70 | header row: 2 | rows: 83,293 | estimated size: ~0.25 GB
   • Cooling_Boiler_Generator_Data_Detail_2023.xlsx | columns: 70 | header row: 2 | rows: 82,933 | estimated size: ~0.24 GB
   • cooling_detail_2015.xlsx | columns: 70 | header row: 2 | rows: 83,209 | estimated size: ~0.13 GB
   • Cooling_Boiler_Generator_Data_Detail_2024.xlsx | columns: 70 | header row: 2 | rows: 82,897 | estimated size: ~0.24 GB
   • cooling_detail_2019.xlsx | columns: 70 | header row: 2 | rows: 84,325 | estimated size: ~0.23 GB
   • cooling_detail_2018.xlsx | columns: 70 | header row: 2 | rows: 84,457 | estimated size: ~0.24 GB
   • cooling_detail_2014.xlsx | columns: 70 | header row: 2 | rows: 83,017 | estimated size: ~0.13 GB
  

In [15]:
df = merge_folder_files(save=True)


🔗 PHASE 2: MERGE
   Source folder: /Users/amyzhang/Documents/Projects_2026/eia_2.0/datasets_thermoelectric
   Reading files into memory as full DataFrames.



Merging files: 100%|████████████████████████████| 11/11 [02:49<00:00, 15.44s/it]



📐 Merged DataFrame shape: (917436, 70)
✅ Files merged: 11 of 11

🔟 TOP 10 COLUMNS WITH MISSING VALUES

                                                                          Absent Count  Absent Percentage
\n.1                                                                            917436         100.000000
\n                                                                              917436         100.000000
860 Cooling Type 2                                                              894432          97.492577
Generator Duct Burners?                                                         745188          81.225066
Water Discharge Name                                                            257244          28.039449
Number Operable Cooling Systems                                                 248700          27.108158
Number Operable Boilers                                                         248700          27.108158
Number Operable Generators                      

In [16]:
# Drop columns that are 100% absent/empty values
df.drop(['\n.1', '\n'], axis=1, inplace=True)

## Quick Profile of Sparse Categorical Columns

In [17]:
# Investigate unique values for categorical columns with nearly 100% absent/empty values
print(df['860 Cooling Type 2'].value_counts(dropna=False))
print("\n")
print(df['Generator Duct Burners?'].value_counts(dropna=False))

860 Cooling Type 2
NaN    894432
RI      10512
HT       6036
RC       2604
ON       2112
OT       1260
OC        348
RF         84
HRI        48
Name: count, dtype: int64


Generator Duct Burners?
NaN    745188
Y      123888
N       48360
Name: count, dtype: int64


In [18]:
print(df['860 Cooling Type 1'].value_counts(dropna=False))

860 Cooling Type 1
(RI) Recirculate: Induced Draft      450420
(ON) Once through No Cool Pond       226020
NaN                                   77700
(RC) Recirculate: Cooling Pond        57456
(DC) Dry Cooling                      51060
(RN) Recirculate: Natural Draft       31608
(OC) Once through with Cool Pond      13464
(RF) Recirculate: Forced Draft         3900
(OT) Other                             3144
(HRI) Hybrid: Dry / Induced Draft      2664
Name: count, dtype: int64


## Interpretation & Decision

- Although `860 Cooling Type 2` and `Generator Duct Burners?` are sparse, the non-null values represent meaningful operational characteristics. Retained for now; final decisions after granularity is established.

- `860 Cooling Type 2` contains shorthand codes (likely mapping to `860 Cooling Type 1`). Remapping postponed - SQL lookup tables will handle human-readable labels.

**TL;DR:** Retain both columns. Postpone remapping.

# 3. Preliminary Dataset Polish for SQL Import

### Why SQL?

  1. **Flexible querying** - Wide schema (70 cols) benefits from views/derived tables without repeated Python transforms
  2. **Validation environment** - Secondary place for sanity checks during iterative cleaning
  3. **Disk-based** - Queries pull only needed data; Python reloads full dataset every session

In [39]:
from scripts.sql_pipeline import schema_preview, sql_prep_columns, dtype_audit, apply_sql_dtypes

# ⬇️ RUN THIS AGAIN -- extended table for breakdown of each column + missingness type
- Something that's going to change for SQL
- IT WILL SAVE IMMEDIATELY

In [19]:
schema_preview(df)

🧬 Full Column Schema (name → dtype):

                                                                            dtype
\n \n \n \n \n \nUtility ID                                                 int64
State                                                                      object
Plant Code                                                                  int64
Plant Name                                                                 object
Year                                                                        int64
Month                                                                       int64
Generator ID                                                               object
Boiler ID                                                                  object
Cooling ID                                                                 object
Generator Primary Technology                                               object
Summer Capacity of Steam Turbines (MW)                      

#### 1) Tackle Column Renaming

In [40]:
sql_df = sql_prep_columns(df)


🧬 Final cleaned column names:
   utility_id, state, plant_code, plant_name, year, month, generator_id, boiler_id, cooling_id, generator_primary_technology, summer_capacity_of_steam_turbines_mw, gross_generation_from_steam_turbines_mwh, net_generation_from_steam_turbines_mwh, summer_capacity_associated_with_single_shaft_combined_cycle_units_mw, gross_generation_associated_with_single_shaft_combined_cycle_units_mwh, net_generation_associated_with_single_shaft_combined_cycle_units_mwh, summer_capacity_associated_with_combined_cycle_gas_turbines_mw, gross_generation_associated_with_combined_cycle_gas_turbines_mwh, net_generation_associated_with_combined_cycle_gas_turbines_mwh, fuel_consumption_from_all_fuel_types_mmbtu, fuel_consumption_from_steam_turbines_mmbtu, fuel_consumption_from_single_shaft_combined_cycle_units_mmbtu, fuel_consumption_from_combined_cycle_gas_turbines_mmbtu, coal_consumption_mmbtu, natural_gas_consumption_mmbtu, petroleum_consumption_mmbtu, biomass_consumption_mmbtu

## 2) Tackle Dtypes
- THIS is why we paid attention to Missingness Representations; because they interrupt inferring dtype

In [43]:
audit_df = dtype_audit(sql_df)
audit_df

,column,inferred_dtype,suggested_dtype,numeric_parseable_%
0,utility_id,int64,INTEGER,100.00
1,state,object,TEXT,0.00
2,plant_code,int64,INTEGER,100.00
3,plant_name,object,TEXT,0.00
4,year,int64,INTEGER,100.00
5,month,int64,INTEGER,100.00
6,generator_id,object,TEXT,39.19
7,boiler_id,object,TEXT,38.27
8,cooling_id,object,TEXT,32.14
9,generator_primary_technology,object,TEXT,0.00


## Quick Profile of 'number_operable_' Columns
Examining values to identify why our type inference function is treating these numeric columns as text

In [44]:
# Investigate unique values for 'number_operable_' columns
print(sql_df['number_operable_generators'].value_counts(dropna=False))
print("\n")
print(sql_df['number_operable_boilers'].value_counts(dropna=False))
print("\n")
print(sql_df['number_operable_cooling_systems'].value_counts(dropna=False))

number_operable_generators
not_reported    668736
NaN             248700
Name: count, dtype: int64


number_operable_boilers
not_reported    668736
NaN             248700
Name: count, dtype: int64


number_operable_cooling_systems
not_reported    668736
NaN             248700
Name: count, dtype: int64


## Interpretation & Decision

The 'number_operable_' columns contain no actual data. While the attribute name suggests plant-level granularity, other features—generator, boiler, and cooling IDs—indicate a unit-level granularity.

**These columns will be dropped**, as they are likely artifacts from an internal cross-survey merge and provide no meaningful information for analysis.


In [45]:
sql_df.drop(['number_operable_generators', 'number_operable_boilers', 'number_operable_cooling_systems'], axis=1, inplace=True)

In [46]:
sql_df = apply_sql_dtypes(sql_df, audit_df)

✅ Converted 65 columns


In [48]:
# Confirm dtype conversion
sql_df.dtypes

utility_id                                                                         Int64
state                                                                     string[python]
plant_code                                                                         Int64
plant_name                                                                string[python]
year                                                                               Int64
month                                                                              Int64
generator_id                                                              string[python]
boiler_id                                                                 string[python]
cooling_id                                                                string[python]
generator_primary_technology                                              string[python]
summer_capacity_of_steam_turbines_mw                                             Float64
gross_generation_from

# 4. Persist SQL-Ready, 'Ground Truth' Dataset

In [52]:
from scripts.utils import save_df

In [53]:
# Save DataFrame
save_df(sql_df, "sql_ready_data.csv")

✅ Saved: /Users/amyzhang/Documents/Projects_2026/eia_2.0/data_exports/sql_ready_data.csv


PosixPath('/Users/amyzhang/Documents/Projects_2026/eia_2.0/data_exports/sql_ready_data.csv')